[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fgram-devAI/nlp-ncsr-task3/blob/main/notebooks/00_baseline_ner_bert.ipynb)

# Baseline — NER with BERT (CoNLL-2003)

NCSR Athens, NLP Assignment 3. Faithful port of `NER-BERT.py` to Colab.

**What this notebook does**
- Downloads CoNLL-2003 English via `kagglehub`.
- Fine-tunes `bert-base-uncased` for NER (BIO tags) for 3 epochs.
- Reports token-level (sklearn) + entity-level (seqeval) metrics on validation each epoch and on the test set at the end.

**This is the Q1 baseline, single run.** The 3-run repetition for mean ± stdev lives in a separate notebook produced after brainstorming/planning.

**Runtime**: Switch to a GPU runtime (`Runtime → Change runtime type → T4 GPU` or better). One epoch ≈ 5–15 min on T4/P100.

In [ ]:
# Silence Pylance/pyright stub-strictness for this Colab-only notebook.
# Stub issues from torch / transformers / sklearn / seqeval are not runtime bugs.
# pyright: reportArgumentType=false, reportAttributeAccessIssue=false, reportPrivateImportUsage=false, reportReturnType=false


## 1. Environment check

In [ ]:
!nvidia-smi || echo 'NO GPU — switch to a GPU runtime before continuing.'

## 2. Install dependencies

Colab ships with `torch` and `transformers` already. We add `seqeval` (entity-level metrics) and `kagglehub` (dataset download).

In [ ]:
%pip install -q seqeval kagglehub

## 3. Kaggle credentials → environment

Before running this cell:
1. Open the **key icon** in the Colab left sidebar ("Secrets").
2. Add two secrets and enable notebook access:
   - `KAGGLE_USERNAME` — your Kaggle username.
   - `KAGGLE_KEY` — the API token from `kaggle.json`.

Never paste credentials inline.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('Kaggle credentials loaded for user:', os.environ['KAGGLE_USERNAME'])

## 4. Download CoNLL-2003

In [ ]:
import kagglehub
from pathlib import Path

dataset_path = Path(kagglehub.dataset_download('alaakhaled/conll003-englishversion'))
print('Downloaded to:', dataset_path)
print('Contents:')
for p in sorted(dataset_path.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(dataset_path), '—', p.stat().st_size, 'bytes')

In [ ]:
# Resolve the actual train/valid/test paths inside the downloaded dataset.
# Kaggle layouts can vary across versions — locate them by name.
def find_one(stem):
    hits = [p for p in dataset_path.rglob(f'{stem}.txt')]
    if not hits:
        raise FileNotFoundError(f'Could not find {stem}.txt under {dataset_path}')
    return hits[0]

train_path = find_one('train')
valid_path = find_one('valid')
test_path = find_one('test')
print('train:', train_path)
print('valid:', valid_path)
print('test :', test_path)

## 5. Imports and hyperparameters

In [ ]:
import time

import torch
import torch.optim as optim
from transformers import BertForTokenClassification, BertTokenizerFast
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall
from tqdm.auto import tqdm

EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cpu':
    raise RuntimeError('No GPU available — switch to a GPU runtime before training.')

## 6. Load CoNLL-2003 sentences

In [ ]:
def load_sentences(filepath):
    sentences = []
    tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            if line.startswith('-DOCSTART-') or line.strip() == '':
                if len(tokens) > 0:
                    sentences.append({
                        'tokens': tokens,
                        'pos_tags': pos_tags,
                        'chunk_tags': chunk_tags,
                        'ner_tags': ner_tags,
                    })
                    tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []
            else:
                parts = line.strip().split(' ')
                if len(parts) >= 4:
                    tokens.append(parts[0])
                    pos_tags.append(parts[1])
                    chunk_tags.append(parts[2])
                    ner_tags.append(parts[3])
    if len(tokens) > 0:
        sentences.append({
            'tokens': tokens,
            'pos_tags': pos_tags,
            'chunk_tags': chunk_tags,
            'ner_tags': ner_tags,
        })
    return sentences

print('loading data')
train_sentences = load_sentences(train_path)
valid_sentences = load_sentences(valid_path)
test_sentences = load_sentences(test_path)
print(f'train={len(train_sentences)}, valid={len(valid_sentences)}, test={len(test_sentences)}')

## 7. Label vocabulary

In [ ]:
all_tags = sorted({tag for s in train_sentences for tag in s['ner_tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)

## 8. Tokenizer + label alignment

In [ ]:
bert_version = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(bert_version)

def align_label(tokens, labels):
    word_ids = tokens.word_ids()
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id.get(labels[word_idx], -100))
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx
    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt',
    )
    labels = align_label(encodings, sentence['ner_tags'])
    return {
        'input_ids': encodings['input_ids'].squeeze(0),
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long),
    }

print('encoding data')
train_dataset = [encode(s) for s in train_sentences]
valid_dataset = [encode(s) for s in valid_sentences]
test_dataset = [encode(s) for s in test_sentences]

class InputDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = InputDataset(train_dataset)
valid_dataset = InputDataset(valid_dataset)
test_dataset = InputDataset(test_dataset)

## 9. Model + optimizer + dataloaders

In [ ]:
print('initializing the model')
model = BertForTokenClassification.from_pretrained(
    bert_version,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
optimizer = optim.AdamW(params=model.parameters(), lr=LR)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

## 10. Evaluation + reporting helpers

In [ ]:
def EvaluateModel(model, data_loader):
    model.eval()
    Y_actual_flat, Y_preds_flat = [], []
    y_true_tags, y_pred_tags = [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            for idx in range(batch['labels'].size(0)):
                true_values_all = batch['labels'][idx]
                mask = (true_values_all != -100)
                true_values = true_values_all[mask]
                pred_values = preds[idx][mask]

                Y_actual_flat.append(true_values)
                Y_preds_flat.append(pred_values)

                y_true_tags.append([id2label[i] for i in true_values.tolist()])
                y_pred_tags.append([id2label[i] for i in pred_values.tolist()])

    Y_actual_flat = torch.cat(Y_actual_flat).detach().cpu().numpy()
    Y_preds_flat = torch.cat(Y_preds_flat).detach().cpu().numpy()
    return Y_actual_flat, Y_preds_flat, y_true_tags, y_pred_tags

def report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name):
    print(f'\n=== {split_name} — Token-level metrics ===')
    print('Accuracy          : {:.3f}'.format(accuracy_score(Y_actual, Y_preds)))
    print('Balanced accuracy : {:.3f}'.format(balanced_accuracy_score(Y_actual, Y_preds)))

    print(f'\n=== {split_name} — Entity-level metrics (seqeval) ===')
    print('Precision : {:.3f}'.format(seqeval_precision(y_true_tags, y_pred_tags)))
    print('Recall    : {:.3f}'.format(seqeval_recall(y_true_tags, y_pred_tags)))
    print('F1        : {:.3f}'.format(seqeval_f1(y_true_tags, y_pred_tags)))

## 11. Training loop

In [ ]:
print('training the model')
training_start = time.perf_counter()

for epoch in range(EPOCHS):
    model.train()
    print(f'epoch {epoch + 1}/{EPOCHS}')
    for batch in tqdm(train_loader, desc=f'Training epoch {epoch + 1}'):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, valid_loader)
    report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags,
                   split_name=f'Validation (epoch {epoch + 1})')

training_seconds = time.perf_counter() - training_start
print(f'\nTotal training time: {training_seconds:.1f}s ({training_seconds/60:.1f} min)')

## 12. Final test-set evaluation

In [ ]:
print('\napplying the model to the test set')
Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, test_loader)

report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name='Test')

label_ids_sorted = list(range(num_labels))
target_names = [id2label[i] for i in label_ids_sorted]
print('\n=== Test — Token-level classification report (sklearn) ===')
print(classification_report(
    Y_actual, Y_preds,
    labels=label_ids_sorted,
    target_names=target_names,
    zero_division=0,
))

print('=== Test — Entity-level classification report (seqeval) ===')
print(seqeval_report(y_true_tags, y_pred_tags, digits=3, zero_division=0))

## 13. (Optional) Persist a metrics JSON

This is the rough shape future notebooks will write to `results/`. For the baseline single
run, downloading the JSON locally + committing manually is fine. The 3-run notebook will
automate this.

In [ ]:
import json, datetime, subprocess

def safe_run(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception:
        return None

metrics = {
    'question': 'Q1-baseline',
    'notebook': '00_baseline_ner_bert.ipynb',
    'model': bert_version,
    'task': 'ner',
    'seed': None,
    'run_index': 0,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'training_seconds': training_seconds,
    'test': {
        'token_micro_accuracy': float(accuracy_score(Y_actual, Y_preds)),
        'token_macro_accuracy': float(balanced_accuracy_score(Y_actual, Y_preds)),
        'entity_micro_f1': float(seqeval_f1(y_true_tags, y_pred_tags, average='micro')),
        'entity_macro_f1': float(seqeval_f1(y_true_tags, y_pred_tags, average='macro')),
    },
    'colab_gpu': safe_run('nvidia-smi --query-gpu=name --format=csv,noheader | head -1'),
    'timestamp_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
}
out_path = '/content/q1_baseline_run.json'
with open(out_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics to', out_path)
print(json.dumps(metrics, indent=2))